In [ ]:
import os
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from collections import deque, Counter

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


In [ ]:

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)


In [ ]:

dataset_dir = "asl_alphabet_train/asl_alphabet_train"


In [ ]:

data = []

for label in os.listdir(dataset_dir):

    label_path = os.path.join(dataset_dir, label)

    if not os.path.isdir(label_path):
        continue

    print(f"Processing {label}...")

    for image_name in os.listdir(label_path):

        image_path = os.path.join(label_path, image_name)

        image = cv2.imread(image_path)

        if image is None:
            continue

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        results = hands.process(image_rgb)

        if results.multi_hand_landmarks:

            for hand_landmarks in results.multi_hand_landmarks:

                landmarks = []

                base_x = hand_landmarks.landmark[0].x
                base_y = hand_landmarks.landmark[0].y
                base_z = hand_landmarks.landmark[0].z

                for lm in hand_landmarks.landmark:

                    landmarks.extend([
                        lm.x - base_x,
                        lm.y - base_y,
                        lm.z - base_z
                    ])

                if len(landmarks) == 63:

                    landmarks.append(label)

                    data.append(landmarks)

print("Feature Extraction Completed")




In [ ]:

columns = [f"feature_{i}" for i in range(63)]
columns.append("label")

df = pd.DataFrame(data, columns=columns)

print(df.head())
print("Dataset Shape:", df.shape)


In [ ]:

df.to_csv("dataset.csv", index=False)

print("CSV Dataset Saved Successfully")


In [ ]:

df.info()


In [ ]:


# Create summary table from df.info()
info_df = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str),
    "Non-Null Count": df.notnull().sum().values
})

# Show first 7 rows
info_show = info_df.head(7)

fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('off')

table = ax.table(
    cellText=info_show.values,
    colLabels=info_show.columns,
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)

plt.savefig("df_info_table.png", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:

df.describe()


In [ ]:


# Describe data
desc = df.describe()

# Show first 7 columns only
desc_show = desc.iloc[:, :7]

fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')

table = ax.table(
    cellText=desc_show.round(2).values,
    rowLabels=desc_show.index,
    colLabels=desc_show.columns,
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)

plt.savefig("df_describe_table.png", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:

df.isnull().sum()


In [ ]:


# Null values count
null_df = pd.DataFrame({
    "Column": df.columns,
    "Null Values": df.isnull().sum().values
})

# Show first 7 rows
null_show = null_df.head(7)

fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('off')

table = ax.table(
    cellText=null_show.values,
    colLabels=null_show.columns,
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)

plt.savefig("null_values_table.png", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:

plt.figure(figsize=(14,6))

sns.countplot(x=df['label'])

plt.title("ASL Alphabet Distribution")
plt.xticks(rotation=90)
plt.show()


In [ ]:

plt.figure(figsize=(14,10))

corr = df.iloc[:, :20].corr()

sns.heatmap(corr, cmap="coolwarm")

plt.title("Feature Correlation Heatmap")
plt.show()


In [ ]:

X = df.drop("label", axis=1)
y = df["label"]

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)


In [ ]:

from sklearn.metrics import accuracy_score

train_acc = []
val_acc = []
iterations = range(1, 11)

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=1,
    warm_start=True,
    random_state=42
)

for i in iterations:
    mlp.fit(X_train, y_train)

    train_pred = mlp.predict(X_train)
    val_pred = mlp.predict(X_test)

    train_accuracy = accuracy_score(y_train, train_pred)
    val_accuracy = accuracy_score(y_test, val_pred)

    train_acc.append(train_accuracy)
    val_acc.append(val_accuracy)

    print(f"Iteration {i}")
    print(f"  Train Accuracy: {train_accuracy:.4f}")
    print(f"  Validation Accuracy: {val_accuracy:.4f}")

print("\nModel Training Completed")


In [ ]:

acc_df = pd.DataFrame({
    "Iteration": list(iterations),
    "Train Accuracy": [round(a, 4) for a in train_acc],
    "Validation Accuracy": [round(a, 4) for a in val_acc]
})

print(acc_df.to_string(index=False))


In [ ]:


# Create accuracy DataFrame
acc_df = pd.DataFrame({
    "Iteration": list(iterations),
    "Train Accuracy": [round(a, 4) for a in train_acc],
    "Validation Accuracy": [round(a, 4) for a in val_acc]
})

# Show table
fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('off')

table = ax.table(
    cellText=acc_df.values,
    colLabels=acc_df.columns,
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)

plt.savefig("accuracy_table.png", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:

plt.figure(figsize=(10, 5))

plt.plot(list(iterations), train_acc, marker='o', label='Train Accuracy', color='blue')
plt.plot(list(iterations), val_acc, marker='s', label='Validation Accuracy', color='orange')

plt.title("Train vs Validation Accuracy per Iteration")
plt.xlabel("Iteration")
plt.ylabel("Accuracy")
plt.xticks(list(iterations))
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

y_pred = mlp.predict(X_test)


In [ ]:

accuracy = accuracy_score(y_test, y_pred)

print("Final Test Accuracy:", accuracy)


In [ ]:

report = classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
)

print(report)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

# Generate classification report as dictionary
report_dict = classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_,
    output_dict=True
)

# Convert to DataFrame
report_df = pd.DataFrame(report_dict).transpose()

# Show first 7 rows
report_show = report_df.head(7)

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

table = ax.table(
    cellText=report_show.round(2).values,
    rowLabels=report_show.index,
    colLabels=report_show.columns,
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)

plt.savefig("classification_report_table.png", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(14,12))

sns.heatmap(cm, annot=False, cmap="Blues")

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


In [ ]:


# take first 10 classes (optional)
labels = sorted(set(y_test))[:10]

cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels,
            yticklabels=labels)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.savefig("confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:

plt.figure(figsize=(8,5))

plt.plot(mlp.loss_curve_)

plt.title("MLP Training Loss Curve")
plt.xlabel("Iterations")
plt.ylabel("Loss")

plt.show()


In [ ]:

def predict_image(image_path):

    image = cv2.imread(image_path)

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    results = hands.process(image_rgb)

    if results.multi_hand_landmarks:

        for hand_landmarks in results.multi_hand_landmarks:

            landmarks = []

            base_x = hand_landmarks.landmark[0].x
            base_y = hand_landmarks.landmark[0].y
            base_z = hand_landmarks.landmark[0].z

            for lm in hand_landmarks.landmark:

                landmarks.extend([
                    lm.x - base_x,
                    lm.y - base_y,
                    lm.z - base_z
                ])

            sample = np.array(landmarks).reshape(1, -1)

            sample = scaler.transform(sample)

            prediction = mlp.predict(sample)

            label = encoder.inverse_transform(prediction)

            print("Predicted Letter:", label[0])

# Example
# predict_image("test.jpg")


In [ ]:
## Save model, scaler, encoder after training (add this in notebook)

import joblib

joblib.dump(mlp,     "model.pkl")
joblib.dump(scaler,  "scaler.pkl")
joblib.dump(encoder, "encoder.pkl")

print("Saved successfully")